In [0]:
%python
df_classfication = spark.sql("""
WITH exclusions AS (
  SELECT *
  FROM dgconfig.public.pii_exclusions
  WHERE is_active = true
),

cols AS (
  SELECT
    c.table_catalog,
    c.table_schema,
    c.table_name,
    c.column_name,
    c.data_type,

    lower(c.table_catalog) AS catalog_l,
    lower(c.table_schema)  AS schema_l,
    lower(c.table_name)    AS table_l,
    lower(c.column_name)   AS column_l,

    concat(
      '_',
      regexp_replace(lower(c.column_name), '[^a-z0-9]+', '_'),
      '_'
    ) AS column_token,

    concat(
      '_',
      regexp_replace(lower(concat_ws('_', c.table_schema, c.table_name)), '[^a-z0-9]+', '_'),
      '_'
    ) AS object_token

  FROM system.information_schema.columns c
),

filtered AS (
  SELECT c.*
  FROM cols c
  LEFT JOIN exclusions e
    ON (e.table_catalog IS NULL OR lower(e.table_catalog) = c.catalog_l)
   AND (e.table_schema  IS NULL OR lower(e.table_schema)  = c.schema_l)
   AND (e.table_name    IS NULL OR lower(e.table_name)    = c.table_l)
   AND (e.column_name   IS NULL OR lower(e.column_name)   = c.column_l)
  WHERE e.exclusion_id IS NULL
),

rule_classified AS (
  SELECT
    *,

    CASE
      WHEN column_token RLIKE '_(pan|pan_no|pan_number|pan_card|pan_id|customer_pan|borrower_pan|applicant_pan)_'
        THEN true
      ELSE false
    END AS is_identity_spine,

    CASE
      -- PAN as separate highest-risk identity spine classification
      WHEN column_token RLIKE '_(pan|pan_no|pan_number|pan_card|pan_id|customer_pan|borrower_pan|applicant_pan)_'
        THEN 'PAN'

      -- Strong sensitive identifiers
      WHEN column_token RLIKE '_(aadhaar|aadhar|uidai|vid|passport|voter_id|driving_license|dl_no|tax_id|gstin|ssn)_'
        THEN 'Sensitive Personal Data'

      -- Financial identifiers
      WHEN column_token RLIKE '_(account_no|bank_account|ifsc|card_no|credit_card|debit_card|cvv|upi|iban|swift)_'
        THEN 'Sensitive Personal Data'

      -- Financial profile
      WHEN column_token RLIKE '_(salary|income|credit_score|loan_amount|emi|premium|net_worth)_'
        THEN 'Sensitive Personal Data'

      -- Direct personal identifiers
      WHEN column_token RLIKE '_(email|mobile|phone|contact_no|dob|date_of_birth|birth_date|gender|age)_'
        THEN 'Personal Data'

      -- Address fields
      WHEN column_token RLIKE '_(address|street|city|state|postcode|pincode|zip|district|region)_'
        THEN 'Personal Data'

      -- Person/customer names only when object context is people/customer/user-like
      WHEN column_token RLIKE '_(first_name|firstname|last_name|lastname|full_name|customer_name|user_name|username)_'
        THEN 'Personal Data'

      WHEN column_token RLIKE '_(name)_'
       AND object_token RLIKE '_(customer|user|person|employee|patient|member|author)_'
        THEN 'Personal Data'

      -- Customer/user IDs are personal only in customer/user/person context
      WHEN column_token RLIKE '_(customer_id|cust_id|user_id|person_id|employee_id|patient_id|member_id|author_id)_'
        THEN 'Personal Data'

      -- Online identifiers
      WHEN column_token RLIKE '_(ip_address|device_id|cookie_id|session_id|advertising_id|gaid|idfa)_'
        THEN 'Personal Data'

      ELSE NULL
    END AS rule_dpdp_classification,

    CASE
      WHEN column_token RLIKE '_(pan|pan_no|pan_number|pan_card|pan_id|customer_pan|borrower_pan|applicant_pan)_'
        THEN 0.99

      WHEN column_token RLIKE '_(aadhaar|aadhar|uidai|passport|tax_id|gstin|ssn|account_no|bank_account|card_no|cvv|upi)_'
        THEN 0.95

      WHEN column_token RLIKE '_(email|mobile|phone|dob|date_of_birth|address|pincode|customer_id|user_id|firstname|lastname|customer_name|ip_address|device_id)_'
        THEN 0.90

      WHEN column_token RLIKE '_(name|id|city|state|region|age|gender)_'
        THEN 0.75

      ELSE 0.50
    END AS rule_confidence

  FROM filtered
),

ai_classified AS (
  SELECT
    *,

    CASE
      WHEN rule_dpdp_classification IS NULL THEN
        ai_classify(
          concat(
            'Classify this column for India DPDP data discovery. ',
            'Do not mark product names, ratings, order dates, technical metadata, or pricing fields as personal data unless they identify a person. ',
            'Context: catalog=', table_catalog,
            ', schema=', table_schema,
            ', table=', table_name,
            ', column=', column_name,
            ', data_type=', data_type
          ),
          array('Personal Data', 'Sensitive Personal Data', 'Non-Personal Data')
        )
      ELSE NULL
    END AS ai_dpdp_classification

  FROM rule_classified
)

SELECT
  table_catalog,
  table_schema,
  table_name,
  column_name,
  --column_token,
  md5(lower(concat(
  table_catalog, '_', table_schema, '_', table_name, '_', column_token, '_', data_type
))) AS column_hash,
  data_type,

  coalesce(rule_dpdp_classification, ai_dpdp_classification, 'Non-Personal Data')
    AS classification_desc,
  CASE
    WHEN classification_desc IN ('Sensitive Personal Data', 'Personal Data')
        THEN 1
    ELSE 0
END AS classification_code,
  is_identity_spine,

  CASE
    WHEN rule_dpdp_classification IS NOT NULL THEN 'RULE'
    WHEN ai_dpdp_classification IS NOT NULL THEN 'AI'
    ELSE 'DEFAULT'
  END AS detection_method,

  CASE
    WHEN rule_dpdp_classification IS NOT NULL THEN rule_confidence
    WHEN ai_dpdp_classification = 'Sensitive Personal Data' THEN 0.70
    WHEN ai_dpdp_classification = 'Personal Data' THEN 0.65
    ELSE 0.60
  END AS confidence_score,

  CASE
    WHEN rule_dpdp_classification = 'PAN' THEN 'AUTO_APPROVED'
    WHEN rule_dpdp_classification IS NOT NULL AND rule_confidence >= 0.70 THEN 'AUTO_APPROVED'
    WHEN ai_dpdp_classification = 'Sensitive Personal Data'
         AND confidence_score >= 0.85
        THEN 'REVIEW_REQUIRED'
    WHEN ai_dpdp_classification = 'Non-Personal Data' THEN 'NA'
    ELSE 'LOW_PRIORITY_REVIEW'
  END AS validation_status,
  current_timestamp() as scanned_ts,
  CASE
    WHEN classification_desc IN ('Sensitive Personal Data', 'Personal Data')
    THEN 'PENDING'
    ELSE 'NA'
  END
  as tag_status,
  NULL as tagged_ts,
  current_timestamp() as created_ts,
  current_timestamp() as updated_ts
FROM ai_classified;
""")

In [0]:
%python
from delta.tables import DeltaTable

target_table = "datagov.default.column_scan_metadata"
deltaTable = DeltaTable.forName(spark, target_table)
merge_output = (deltaTable.alias("tgt").merge(
    df_classfication.alias("src"),
        """
        tgt.column_hash = src.column_hash
        """
    ).whenNotMatchedInsertAll() \
    .whenNotMatchedBySourceDelete() \
    .execute())
display(merge_output)

In [0]:
BEGIN
  FOR r AS
    SELECT concat(
      'SET TAG ON COLUMN ',
      m.table_catalog, '.', m.table_schema, '.', m.table_name, '.', m.column_name,
      ' dpdp_class = personal_data'
    ) AS sql_cmd
    FROM datagov.default.column_scan_metadata m
    WHERE m.validation_status = 'AUTO_APPROVED'
      AND m.tag_status = 'PENDING'
      AND NOT EXISTS (
        SELECT 1
        FROM system.information_schema.column_tags t
        WHERE t.catalog_name = m.table_catalog
          AND t.schema_name  = m.table_schema
          AND t.table_name   = m.table_name
          AND t.column_name  = m.column_name
          AND t.tag_name     = 'dpdp_class'
      )
  DO
    BEGIN
      DECLARE EXIT HANDLER FOR SQLEXCEPTION
      BEGIN
        DECLARE err_message STRING;
        GET DIAGNOSTICS CONDITION 1 err_message = MESSAGE_TEXT;
        INSERT INTO datagov.default.tag_errors
        VALUES (
          current_timestamp(),
          r.sql_cmd,
          err_message
        );
      END;
      EXECUTE IMMEDIATE r.sql_cmd;
    END;
  END FOR;
END;

In [0]:
UPDATE datagov.default.column_scan_metadata
SET tag_status = 'TAGGED',
    tagged_ts = current_timestamp(),
    updated_ts = current_timestamp()
WHERE validation_status = 'AUTO_APPROVED'
  AND tag_status = 'PENDING'
  AND EXISTS (
    SELECT 1
    FROM system.information_schema.column_tags t
    WHERE t.catalog_name = table_catalog
      AND t.schema_name  = table_schema
      AND t.table_name   = table_name
      AND t.column_name  = column_name
      AND t.tag_name     = 'dpdp_class'
  );

In [0]:
select * from datagov.default.column_scan_metadata